### Close CRM - Source Data Exploration
This notebook documents initial source-data exploration used to inform webhook ingestion and lead enrichment design. Original source payloads are excluded from the public repository to protect potentially sensitive information. Key findings and resulting design decisions are summarized in docs/03_eda_findings.md.

In [ ]:
# Imports and Setup

import json
from pathlib import Path

data_dir = Path("../sample_data/close")

with open(data_dir / "close_crm_event.json", 'r') as file:
    crm_event = json.load(file)

with open(data_dir / "lead_owner_lookup.json", 'r') as file:
    lead_owner = json.load(file)

In [ ]:
print(crm_event.keys())
print(lead_owner.keys())

In [ ]:
# webhook metadata
crm_event["event"].keys()

In [ ]:
# crm record with lead attributes
crm_event["event"]["data"].keys()

In [ ]:
# lead_id as join key 

crm_lead_id = crm_event["event"]["lead_id"]
lead_owner_lead_id = lead_owner["lead_id"]

print("CRM lead ID:", crm_lead_id)
print("Lead-owner lead ID", lead_owner_lead_id)
print("IDs match:", crm_lead_id == lead_owner_lead_id)

In [ ]:
# lead and lead-owner payloads

close_crm_lead = crm_event["event"]["data"]

print("FROM CLOSE CRM PAYLOAD")
print("Name:", close_crm_lead["display_name"])
print("Status:", close_crm_lead["status_label"])
print("Created:", close_crm_lead["date_created"])

print("\nFROM LEAD-OWNER LOOKUP PAYLOAD")
print("Email:", lead_owner["lead_email"])
print("Owner:", lead_owner["lead_owner"])
print("Funnel:", lead_owner["funnel"])

In [ ]:
# Id fields shared by both crm payload and lead owner payload

shared_fields = close_crm_lead.keys() & lead_owner.keys()

print("SHARED FIELDS")
for field in sorted(shared_fields):
    print(field)

In [ ]:
print("SHARED FIELD VALUES")

for field in sorted(shared_fields):
    print(f"\n{field}")
    print("Close CRM:", close_crm_lead[field])
    print("Lead-owner lookup:", lead_owner[field])
    print("Match:", close_crm_lead[field] == lead_owner[field])


In [ ]:
# lead_id is static, event_id retry should retain same event_id --> avoid duplicate notifications on same, new lead set

close_event = crm_event["event"]

print("CLOSE CRM EVENT METADATA")
print("Event ID:", close_event["id"])
print("Lead ID:", close_event["lead_id"])
print("Action:", close_event["action"])
print("Object type:", close_event["object_type"])
print("Event created:", close_event["date_created"])
print("Lead created:", close_crm_lead["date_created"])

In [ ]:
# condition to process lead

is_lead_created = (
    close_event["object_type"] == 'lead' and
    close_event["action"] == 'created'
)

is_lead_created

In [ ]:
print("Event created:", close_event["date_created"])
print("Lead created:", close_crm_lead["date_created"])

In [ ]:
custom_fields = {
    field: value
    for field, value in close_crm_lead.items()
    if field.startswith("custom.")
}

print("CLOSE CRM CUSTOM FIELDS")

for field, value in custom_fields.items():
    print(f"{field}: {value}")